# 03 - Transaction Preprocessing & Modeling Readiness

Prepare complete order baskets for Phase 2 without executing Apriori or FP-Growth. Explain feature types, scaling decisions, and record-level data quality.

The processed tables are generated from a reproducible random sample of 30,000 prior Instacart orders. All product lines belonging to each selected order are retained, preserving complete transaction baskets.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# Original Instacart, 30,000 reproducibly selected prior orders.

DATA_MODE = 'real'
print('DATA MODE:', DATA_MODE.upper(), '|', 'REAL INSTACART SAMPLE' if DATA_MODE == 'real' else 'SYNTHETIC DEMO')

DATA MODE: REAL | REAL INSTACART SAMPLE


## Ensure reproducible processed tables

Each basket stores a unique product-ID list, delimited by `|`, derived from complete selected prior orders. `user_id` is a customer reference, not a numerical quantity.
The cleaning report records the sampling method, selected order count, and random seed.

In [2]:
import pandas as pd
import json
from IPython.display import display

processed = ROOT / 'data/processed'

# To regenerate: python -m scripts.run_phase1 --max-orders 30000

baskets = pd.read_csv(processed / 'baskets.csv')
order_products = pd.read_csv(processed / 'order_products_clean.csv')
product_catalog = pd.read_csv(processed / 'product_catalog.csv')

report = json.loads((processed / 'cleaning_report.json').read_text())

print('Selected prior orders:', report['selected_prior_orders'])
print('Random seed:', report['random_seed'])
print('Sampling method:', report['sampling_method'])

print('Unique baskets:', baskets['order_id'].nunique())
print('Empty baskets:', int((baskets['basket_size'] == 0).sum()))

display(baskets.head(10))



Selected prior orders: 30000
Random seed: 42
Sampling method: Deterministic random sample of prior order IDs using seed 42. Complete product lines for selected orders are filtered in chunks. This is not a chronological split.
Unique baskets: 30000
Empty baskets: 0


,order_id,user_id,basket_size,product_ids,product_names,order_dow,order_hour_of_day
0,136,184355,4,11005|18465|29926|47766,Organic Cream Cheese Bar | Organic Grade A Fre...,6,10
1,376,121858,7,890|2716|8277|16462|35851|45210|47029,Organic Diced Tomatoes | Organic Greek Lowfat ...,5,16
2,522,35250,20,1862|2344|10251|13292|17706|24184|24852|25340|...,Tuna Fillets in Olive Oil | Organic SprouTofu ...,1,15
3,526,17900,18,1752|1831|1940|9387|12191|12302|22489|23085|27...,Mango Sorbet | Peach-Pear Sparkling Water | Or...,0,23
4,552,18926,4,4091|5268|33452|37011,Kosher Whole Dill Pickles | Bagel Bites Three ...,4,13
5,776,149408,7,1216|3740|18465|18883|26452|28934|31526,Dairy Free Gluten Free Mushroom & Roasted Garl...,0,20
6,873,189247,12,2380|4605|7021|8518|20114|21070|28535|31717|35...,Cayenne Pepper | Yellow Onions | Organic Tomat...,0,10
7,911,172302,11,19457|19822|20084|25653|26882|30459|32931|3298...,Organic One Ingredient Peanut Butter | Kosher ...,2,12
8,1135,201623,22,3957|5134|10017|11737|17600|19816|22035|24852|...,100% Raw Coconut Water | Organic Thompson Seed...,2,7
9,1348,44705,10,5785|6182|8277|13176|15182|22115|36070|39921|4...,Organic Reduced Fat 2% Milk | Natural Artesian...,6,11


In [5]:
# Basket-level integrity
assert baskets['order_id'].is_unique
assert baskets['basket_size'].gt(0).all()
assert baskets['product_ids'].notna().all()

for _, basket in baskets.iterrows():
    basket_products = basket['product_ids'].split('|')

    assert len(basket_products) == len(set(basket_products))
    assert len(basket_products) == basket['basket_size']

    cleaned_products = order_products.loc[
        order_products['order_id'] == basket['order_id'],
        'product_id'
    ].astype(str).tolist()

    assert set(basket_products) == set(cleaned_products)

assert not order_products.duplicated(subset=['order_id', 'product_id']).any()
assert order_products['product_id'].isin(product_catalog['product_id']).all()

print('Basket integrity: PASSED')
print('Unique order-product pairs: PASSED')
print('Product catalog membership: PASSED')
print('Complete transaction preservation: PASSED')

Basket integrity: PASSED
Unique order-product pairs: PASSED
Product catalog membership: PASSED
Complete transaction preservation: PASSED


## Preprocessing justification
- Data for association mining is **categorical transactional presence**; product IDs are labels, not continuous numeric features.
- Do **not** apply StandardScaler or MinMaxScaler to product IDs. Their numerical values do not represent magnitude or distance.
- Do **not** impute absent items as missing values. An absent product means it was not purchased in that transaction.
- Phase 2 may one-hot encode product presence, transforming each transaction into binary item indicators.
- `user_id` is an identifier, not a numerical feature for scaling.
- Reordered status describes historical purchasing behaviour but does not create a supervised target for association-rule mining.
- For evaluation, hold out complete transactions or users before mining rules, preventing leakage between training and evaluation baskets.

In [4]:
print('Product ID examples (categorical):', baskets['product_ids'].head(3).tolist())
print('Scaling: NOT APPLICABLE — product IDs are categorical identifiers ' 'and become binary presence indicators in Phase 2.')

print('Total baskets:', len(baskets))
print('Total cleaned purchase lines:', len(order_products))
print('Products in catalog:', len(product_catalog))
print('Mean basket size:', round(baskets['basket_size'].mean(), 2))
print('Median basket size:', baskets['basket_size'].median())

print('Record-level integrity checks: PASSED')

Product ID examples (categorical): ['11005|18465|29926|47766', '890|2716|8277|16462|35851|45210|47029', '1862|2344|10251|13292|17706|24184|24852|25340|25837|26209|30391|30489|31130|31444|32611|34126|35939|41950|45200|46676']
Scaling: NOT APPLICABLE — product IDs are categorical identifiers and become binary presence indicators in Phase 2.
Total baskets: 30000
Total cleaned purchase lines: 302305
Products in catalog: 49688
Mean basket size: 10.08
Median basket size: 8.0
Record-level integrity checks: PASSED
